# ETL Data HR MNC: CSV to MySQL

This notebook performs the **ETL (Extract, Transform, Load)** process on the [MNC HR Data](https://www.kaggle.com/datasets/rohitgrewal/hr-data-mnc) dataset from Kaggle (approximately 2 million rows), converting it into an **SQL dump file (`.sql`)** that is ready to be imported into MySQL.

| Stage | Description | Cell |
|---|---|---|
| **Extract** | Download the dataset from Kaggle and locate the CSV file | Step 2 |
| **Transform** | Standardize column names to `snake_case` and map pandas data types to MySQL types | Steps 3–4 |
| **Load** | Write `CREATE TABLE` + `INSERT` to a `.sql` file, then compress and download | Steps 4–5 |

**Workflow:**
1. Install libraries
2. Download the dataset from Kaggle
3. Initial exploration and column name cleaning
4. Convert CSV to SQL files (in chunks)
5. Compress to ZIP and download

> **Note:** Because the dataset is large, the CSV files are read **in chunks** to prevent Colab’s RAM from filling up.

## Step 1: Installing Libraries

Required libraries:
- `kagglehub`: downloads datasets directly from Kaggle
- `sqlalchemy` and `pymysql`: MySQL connectors/drivers (required if you want to load data directly into a database later)
- `tqdm`: displays a progress bar during the conversion process

In [1]:
!pip install -q kagglehub sqlalchemy pymysql tqdm

Import Libraries

In [2]:
import kagglehub
import os
import glob
import pandas as pd
from tqdm.notebook import tqdm
import zipfile
from google.colab import files

## Step 2: Extract and Download the Dataset from Kaggle

The dataset is downloaded using `kagglehub`, and then the program searches for `*.csv` files in the download folder. If no CSV files are found, the program stops with an error message. The path to the CSV file is stored in the variable `csv_file_path` for use in the next cell.

In [3]:
# Download latest version
path = kagglehub.dataset_download("rohitgrewal/hr-data-mnc")
print("Path to dataset files:", path)

# Searching for a CSV file in a directory path
csv_files = glob.glob(os.path.join(path, "*.csv"))
if not csv_files:
    raise FileNotFoundError("The CSV file was not found in the dataset folder!")

csv_file_path = csv_files[0]
print("CSV file found:", csv_file_path)

100%|██████████| 66.7M/66.7M [00:00<00:00, 121MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/rohitgrewal/hr-data-mnc/versions/1
CSV file found: /root/.cache/kagglehub/datasets/rohitgrewal/hr-data-mnc/versions/1/HR_Data_MNC_Data Science Lovers.csv


## Step 3: Initial Exploration and Column Name Cleaning

Only the **first 5 rows** are read (`nrows=5`) to conserve RAM. The goals are:
- to view the original column names and sample data
- to define the `clean_column_names()` function, which converts column names to lowercase and `snake_case`—that is, *trims* leading and trailing spaces, converts to lowercase, and replaces spaces, `-`, and `/` with `_`

Well-formatted column names are necessary to ensure they can be safely used as column names in MySQL.

In [4]:
# Read only the first 5 lines to conserve RAM
df_sample = pd.read_csv(csv_file_path, nrows=5)

print("Kolom Asli:")
print(df_sample.columns.tolist())

# A function to reformat column names to conform to the MySQL standard (snake_case)
def clean_column_names(columns):
    return [
        col.strip().lower().replace(" ", "_").replace("-", "_").replace("/", "_")
        for col in columns
    ]

cleaned_columns = clean_column_names(df_sample.columns)
print("\nColumns After Cleaning:")
print(cleaned_columns)

display(df_sample.head(3))

Kolom Asli:
['Unnamed: 0', 'Employee_ID', 'Full_Name', 'Department', 'Job_Title', 'Hire_Date', 'Location', 'Performance_Rating', 'Experience_Years', 'Status', 'Work_Mode', 'Salary_INR']

Columns After Cleaning:
['unnamed:_0', 'employee_id', 'full_name', 'department', 'job_title', 'hire_date', 'location', 'performance_rating', 'experience_years', 'status', 'work_mode', 'salary_inr']


,Unnamed: 0,Employee_ID,Full_Name,Department,Job_Title,Hire_Date,Location,Performance_Rating,Experience_Years,Status,Work_Mode,Salary_INR
0,0,EMP0000001,Joshua Nguyen,IT,Software Engineer,2011-08-10,"Isaacland, Denmark",5,14,Resigned,On-site,1585363
1,1,EMP0000002,Julie Williams,Marketing,SEO Specialist,2018-03-02,"Anthonyside, Costa Rica",2,7,Active,On-site,847686
2,2,EMP0000003,Alyssa Martinez,HR,HR Manager,2023-03-20,"Port Christinaport, Saudi Arabia",1,2,Active,On-site,1430084


## Step 4: Transform and Load—Convert CSV to an SQL File

This cell generates the file `hr_data_mnc.sql` through the following steps:

1. **Configuration**: output file name, table name (`hr_attrition`), chunk size (10,000 rows), and `MAX_ROWS` (enter a number to limit the number of rows; `None` to include all data).
2. **Data type mapping** via `get_mysql_type()`:

   | Pandas | MySQL |
   |---|---|
   | integer | `BIGINT` |
   | float | `DOUBLE` |
   | boolean | `BOOLEAN` |
   | others (text) | `VARCHAR(255)` |

3. A **`CREATE TABLE`** statement is generated from the first 100 sample rows.
4. **`INSERT`** statements are written in stages: the CSV is read in chunks of 10,000 rows, then further split into sub-batches of 1,000 rows per statement to avoid exceeding MySQL’s `max_allowed_packet` limit.
5. Missing values (`NaN`) are converted to `NULL`, while text is *escaped* (using `'` and `\`) to ensure valid SQL.
6. The header temporarily disables `FOREIGN_KEY_CHECKS`, `UNIQUE_CHECKS`, and `AUTOCOMMIT` to speed up the import process, then re-enables them at the end of the file.

> This process takes time because the data consists of approximately 2 million rows.

In [5]:
output_sql_file = "hr_data_mnc.sql"
table_name = "hr_attrition"
chunk_size = 10000      # Number of rows per pass
MAX_ROWS = None         # Change this to a number (e.g., 100000) if you want to limit the number of rows, or set it to None to include all data

# Mapping Pandas Data Types to MySQL
def get_mysql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "BIGINT"
    elif pd.api.types.is_float_dtype(dtype):
        return "DOUBLE"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    else:
        return "VARCHAR(255)"

1. Create a CREATE TABLE statement from the sample data

In [6]:
df_init = pd.read_csv(csv_file_path, nrows=100)
df_init.columns = clean_column_names(df_init.columns)

col_defs = []
for col_name, dtype in df_init.dtypes.items():
    mysql_type = get_mysql_type(dtype)
    col_defs.append(f"    `{col_name}` {mysql_type}")

create_table_sql = f"DROP TABLE IF EXISTS `{table_name}`;\n"
create_table_sql += f"CREATE TABLE `{table_name}` (\n" + ",\n".join(col_defs) + "\n) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;\n\n"

2. Write to the .sql file in stages (chunking)

In [7]:
with open(output_sql_file, "w", encoding="utf-8") as f:
    f.write("-- MySQL Dump Generated from Google Colab\n")
    f.write("SET FOREIGN_KEY_CHECKS = 0;\n")
    f.write("SET UNIQUE_CHECKS = 0;\n")
    f.write("SET AUTOCOMMIT = 0;\n\n")
    f.write(create_table_sql)

    total_written = 0
    cols_formatted = ", ".join([f"`{c}`" for c in df_init.columns])

    # Reading CSV by chunk
    for chunk in tqdm(pd.read_csv(csv_file_path, chunksize=chunk_size, nrows=MAX_ROWS), desc="Mengonversi ke SQL"):
        chunk.columns = clean_column_names(chunk.columns)

        # Sub-batch every 1,000 rows for a single INSERT statement to avoid exceeding MySQL's max_allowed_packet limit
        sub_batch_size = 1000
        for i in range(0, len(chunk), sub_batch_size):
            sub_chunk = chunk.iloc[i:i+sub_batch_size]
            values_list = []

            for row in sub_chunk.itertuples(index=False, name=None):
                escaped_vals = []
                for val in row:
                    if pd.isna(val):
                        escaped_vals.append("NULL")
                    elif isinstance(val, (int, float)):
                        escaped_vals.append(str(val))
                    else:
                        # Escape single quotation marks and backslashes in the text
                        clean_str = str(val).replace("\\", "\\\\").replace("'", "\\'")
                        escaped_vals.append(f"'{clean_str}'")
                values_list.append("(" + ", ".join(escaped_vals) + ")")

            insert_stmt = f"INSERT INTO `{table_name}` ({cols_formatted}) VALUES\n" + ",\n".join(values_list) + ";\n"
            f.write(insert_stmt)

        total_written += len(chunk)

    f.write("\nCOMMIT;\n")
    f.write("SET UNIQUE_CHECKS = 1;\n")
    f.write("SET FOREIGN_KEY_CHECKS = 1;\n")

print(f"\nDone! Total {total_written:,} The line was successfully converted to '{output_sql_file}'.")

Mengonversi ke SQL: 0it [00:00, ?it/s]


Done! Total 2,000,000 The line was successfully converted to 'hr_data_mnc.sql'.


## Step 5: Compress and Download the Results

The `.sql` file is compressed into `hr_data_mnc_sql.zip` to significantly reduce its size, then downloaded to the local computer using `files.download()`.

**How to import into MySQL** after the ZIP file is extracted:
```bash
mysql -u <user> -p <database_name> <hr_data_mnc.sql
```

In [8]:
zip_filename = "hr_data_mnc_sql.zip"

print("Compressing SQL files into a ZIP file...")
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(output_sql_file)

print(f"File ready: {zip_filename} ({os.path.getsize(zip_filename) / (1024*1024):.2f} MB)")

Compressing SQL files into a ZIP file...
File ready: hr_data_mnc_sql.zip (71.14 MB)


Trigger a download to the local computer

In [9]:
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>